In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "nayohan/llama3-instrucTrans-enko-8b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
  model_name,
  device_map="auto",
  torch_dtype=torch.bfloat16
)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from difflib import SequenceMatcher
import time

# 영어 문장 리스트
sentences = ["ok babe",
    "what happend babe",
    "oh man i hate that what kinda of god?",
    "ic i have a pitbull hes a sweet heart",
    "he thinks his a small dog",
    "how often do u shave ur legs?",
    "if he had his way he would but he will streach out and take over the couch",
    "u have hair down there?",
    "want to see his baby pics",
    "u ever have enough to shave it?",
    "woundering where u where and doing baby",
    "how long is ur hair?",
    "I know, Im excited to see u",
    "that would be nice, u ever kiss a boy?",
    "you know what baby your sssoooo allsome",
    "youll be a little heart breaker when your older",
    "yeah i ordered papjohns pizza and u baby",
    "didnt know you were only 13 but its ok",
    "ok babby what u eatting ?",
    "send me the sexy pic",
    "yah my day was good",
    "had a couple beers and was buzzing for a little bit",
    "it really turns me on.",
    "email me when u have time to get together...",
    "not married.. im divorced 5 years"
]

# 정답 번역 리스트 (예시 - 실제 번역 기준으로 수정해줘야 정확함)
references = ["좋아, 자기야",                            # ok babe
    "무슨 일이야, 자기야?",                       # what happend babe
    "아, 이런 거 정말 싫어… 이런 걸 만든 신이 대체 누구야?",      # oh man i hate that what kinda of god?
    "알겠어, 나 핏불 키워. 정말 상냥한 녀석이야.",          # ic i have a pitbull hes a sweet heart
    "그는 자신이 작은 강아지라고 생각해.",            # he thinks his a small dog
    "다리 얼마나 자주 밀어?",                       # how often do u shave ur legs?
    "만약 마음대로 할 수 있다면, 그는 드러눕고, 소파를 다 차지할 거야.",    # if he had his way he would but he will streach out and take over the couch
    "거기 털 있어?",                            # u have hair down there?
    "그의 아기 때 사진 보고 싶어?",                       # want to see his baby pics
    "그걸 깎을 만큼 충분히 길었던 적 있어?",                 # u ever have enough to shave it?
    "네가 어디서 뭘 하고 있는지 궁금해, 자기야.",               # woundering where u where and doing baby
    "머리카락 길이가 얼마나 돼?",                    # how long is ur hair?
    "알아, 너 만나는 거 기대돼.",                    # I know, Im excited to see u
    "그러면 좋겠네요, 남자와 키스해 본 적 있나요?",                # that would be nice, u ever kiss a boy?
    "있잖아, 자기야, 넌 정말 대단해.",                   # you know what baby your sssoooo allsome
    "네가 크면 사람 마음 많이 아프게 할 거야.",             # youll be a little heart breaker when your older
    "응, 파파존스 피자를 주문했어… 그리고 네 생각 중이야, 자기야.",  # yeah i ordered papjohns pizza and u baby
    "당신이 겨우 13살인 줄 몰랐지만 괜찮습니다",               # didnt know you were only 13 but its ok  (미성년자 언급 → 빈칸)
    "오케이, 자기야, 뭐 먹고 있어?",                     # ok babby what u eatting ?
    "나한테 섹시한 사진 보내줘",                         # send me the sexy pic  (노골적 성적 요청 → 빈칸)
    "응, 내 하루는 괜찮았어.",                        # yah my day was good
    "맥주 몇 잔 마셨더니 잠깐 취했어.",                  # had a couple beers and was buzzing for a little bit
    "그거 정말 흥분돼",                                      # it really turns me on.  (노골적 성적 암시 → 빈칸)
    "시간 되면 모임 잡자고 이메일 줘.",                 # email me when u have time to get together...
    "결혼 안 했어.. 이혼한 지 5년 됐어."                   # not married.. im divorced 5 years
]

# 번역 결과 저장
translated = []

# 성능 통계 저장
bleu_scores = []
word_overlaps = []
char_overlaps = []
length_ratios = []

start_time = time.time()

for src in sentences:
    conversation = [{'role': 'system', 'content': "당신은 번역기 입니다. 영어를 한국어로 번역하세요."},
                    {'role': 'user', 'content': src}]
    inputs = tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to("cuda")

    attention_mask = inputs != tokenizer.pad_token_id

    outputs = model.generate(
        inputs,
        attention_mask=attention_mask,
        max_new_tokens=4096,
        pad_token_id=tokenizer.eos_token_id  # 경고 해결
    )
    output_text = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    translated.append(output_text.strip())

# 개별 성능 측정
for hyp, ref in zip(translated, references):
    bleu = sentence_bleu([ref.split()], hyp.split())  # 간단한 BLEU
    word_overlap = len(set(hyp.split()) & set(ref.split())) / len(set(ref.split()))
    char_overlap = SequenceMatcher(None, hyp, ref).ratio()
    length_ratio = len(hyp) / len(ref) if len(ref) > 0 else 0

    bleu_scores.append(bleu)
    word_overlaps.append(word_overlap)
    char_overlaps.append(char_overlap)
    length_ratios.append(length_ratio)

end_time = time.time()
total_time = end_time - start_time

# 평균 계산
avg_metrics = {
    'bleu_score': sum(bleu_scores) / len(bleu_scores),
    'word_overlap': sum(word_overlaps) / len(word_overlaps),
    'char_overlap': sum(char_overlaps) / len(char_overlaps),
    'length_ratio': sum(length_ratios) / len(length_ratios)
}

# 출력
print(f"\n📈 전체 성능 통계")
print("=" * 100)
print(f"평균 BLEU Score: {avg_metrics['bleu_score']:.4f}")
print(f"평균 단어 겹침률: {avg_metrics['word_overlap']:.4f}")
print(f"평균 문자 겹침률: {avg_metrics['char_overlap']:.4f}")
print(f"평균 길이 비율: {avg_metrics['length_ratio']:.4f}")
print(f"총 번역 시간: {total_time:.2f}초")
print(f"평균 번역 시간: {total_time/len(sentences):.2f}초/문장")


In [ ]:
# 📈 전체 성능 통계
# =========================
# 평균 BLEU Score: 0.0000
# 평균 단어 겹침률: 0.2554
# 평균 문자 겹침률: 0.5591
# 평균 길이 비율: 1.0750
# 총 번역 시간: 288.42초
# 평균 번역 시간: 11.54초/문장